# Basis Transforms

Transform vector and tensor components between curvilinear and Cartesian
bases, then check that the transforms can be undone.

Navigation: [Index][index] | Previous: [Reference-Metric Applications][prev]
| Next: [Curvilinear Wave Equation][next]

[index]: ../index.ipynb
[prev]: reference_metric_applications.ipynb
[next]: ../3-wave_equation/wave_equation_curvilinear.ipynb

## Learning Goals

- Transform vector and tensor components between bases.
- Separate a physical vector from the numbers used to describe it.
- Use round-trip residuals to check that a transform can be undone.

## Words for This Notebook

- **Basis:** the local directions used to describe components of a vector or
  tensor.
- **Component:** one number in a vector or tensor description.
- **Reference metric:** the coordinate-system geometry used as the baseline
  for curvilinear calculations.
- **Jacobian:** a table of derivatives that converts between coordinate
  labels.
- **Round trip:** transforming to another basis and then back again.
- **Residual:** the leftover after subtracting the expected expression.

Use the code cells actively: predict the result, run the cell, then explain
the output in plain language.

## Notation

Spherical coordinates map to Cartesian coordinates as

$$
x = r \sin(\theta)\cos(\phi), \quad
y = r \sin(\theta)\sin(\phi), \quad
z = r \cos(\theta).
$$

A contravariant vector transforms with the Jacobian:

$$
V^i_{\rm Cart} = \frac{\partial x^i}{\partial q^j} V^j_{\rm rfm}.
$$

The validation cells below subtract the original components after a round
trip. A zero residual means the symbolic transform matched the identity.

## Step 1: Import Tools

These imports provide SymPy and the NRPy modules used below.

In [1]:
import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.reference_metric as refmetric
from nrpy.equations.basis_transforms.jacobians import BasisTransforms

## Step 2: Build the Spherical Reference Metric

The output connects the coordinate map to the metric scale factors.

In [2]:
spherical = refmetric.reference_metric["Spherical"]
transforms = BasisTransforms("Spherical")
print("reference metric:", spherical.CoordSystem)
print("scale factors:", spherical.scalefactor_orthog)

Setting up reference_metric[Spherical]...


reference metric: Spherical
scale factors: [1, xx0, xx0*sin(xx1)]


## Step 3: Transform a Radial Vector

The next cell represents a vector with only a radial component. In Cartesian
coordinates, its components depend on direction.

In [3]:
radial_amplitude = sp.Symbol("V_r", real=True)
radial_vectorU = [radial_amplitude, sp.sympify(0), sp.sympify(0)]
cartesian_vectorU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    radial_vectorU
)
print("Cartesian components of a radial vector:")
for component in cartesian_vectorU:
    print(sp.trigsimp(component))

Cartesian components of a radial vector:
V_r*sin(xx1)*cos(xx2)
V_r*sin(xx1)*sin(xx2)
V_r*cos(xx1)


## Validation Check

First check that a generic vector returns to itself after transforming to
Cartesian components and back.

In [4]:
generic_vectorU = ixp.declarerank1("V")
cartesian_genericU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    generic_vectorU
)
round_trip_vectorU = transforms.basis_transform_vectorU_from_Cartesian_to_rfmbasis(
    cartesian_genericU
)
round_trip_residuals = [
    sp.trigsimp(sp.simplify(round_trip_vectorU[i] - generic_vectorU[i]))
    for i in range(3)
]
print("contravariant vector residuals:", round_trip_residuals)
if round_trip_residuals != [0, 0, 0]:
    raise RuntimeError("Expected vector round-trip residuals to vanish.")

contravariant vector residuals: [0, 0, 0]


The Cartesian metric should reduce to the identity matrix:

$$
\hat{\gamma}^{\rm Cart}_{ij}
= \frac{\partial q^a}{\partial x^i}
  \frac{\partial q^b}{\partial x^j}
  \hat{\gamma}^{\rm rfm}_{ab}.
$$

The next cell checks diagonal entries against `1` and one off-diagonal entry
against `0`.

In [5]:
cartesian_metricDD = transforms.basis_transform_tensorDD_from_rfmbasis_to_Cartesian(
    transforms.rfm.ghatDD
)
diagonal_residuals = [
    sp.trigsimp(sp.simplify(cartesian_metricDD[i][i] - 1)) for i in range(3)
]
off_diagonal_residual = sp.trigsimp(sp.simplify(cartesian_metricDD[0][1]))
print("Cartesian metric diagonal residuals:", diagonal_residuals)
print("Cartesian metric (0, 1) residual:", off_diagonal_residual)
if diagonal_residuals != [0, 0, 0] or off_diagonal_residual != 0:
    raise RuntimeError("Expected transformed metric residuals to vanish.")

Cartesian metric diagonal residuals: [0, 0, 0]
Cartesian metric (0, 1) residual: 0


The transformed components show how vector and tensor data change basis while
representing the same geometric object.

## Learning Check

Before the radial-vector transform, predict whether Cartesian `x`, `y`, and
`z` components should all be nonzero away from the axes.

## Continue to the Curvilinear Wave Equation

- [Reference-Metric Applications](reference_metric_applications.ipynb)
- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb)
- [Relativity Variable Conversions](../6-numerical_relativity/adm_bssn_conversions.ipynb)